# H04 — 让旋钮改变亮度，但别把数字当电压

**学习入口：把下面 prompt 发给 VS Code Codex 侧边栏。** Notebook 也可自行逐格阅读。
C++ 在持续 ESP-IDF 工程编辑与构建；Notebook 运行仅显示示例/手动证据卡。
**状态：Alpha；固件构建与实机验收分开记录，见 ../VALIDATION.md。**

```text
开始 H04 — 让旋钮改变亮度，但别把数字当电压。
请先读取本工程 AGENTS.md、.academy/course.json 和 .academy/progress.md。
通过 course_root 找到 step2/course.json 和 step2/notebooks/H04-measurement.ipynb。
读取 Notebook 的 cells/source 与 academy.step_id/mode；先读当前步骤相关格。
检查工程、环境和前置条件，从第一个未完成步骤开始；每轮只推进一个小步骤。
Guided 可合并指定示例并说明 diff；Modify/Debug/Build/Open 先让我尝试，按需逐级提示。
不要主动读取 solution.md 或其他答案。环境故障可以排查修复，解释后回到原步骤。
不要把挑战 Bug 自动修掉。以当前代码与真实证据验收，记录提示级别和下一步。
我已有一定 C++ 基础，Python 几乎零基础。现在开始。
```

不用 agent 时按步骤标题顺序操作；卡住后再逐级查看本课 hints.md。启动 prompt 的路径按工作区课程链接解析，不靠其他窗口或聊天记忆。

## 我们要解决什么问题？

把桌面提醒器的亮度交给一个旋钮。屏幕上的 ADC 数字、万用表电压和 LED 亮度为什么不是同一件事？

前置：H03；10 kΩ 电位器，万用表可选。

## 本次够用的知识

PWM 的占空比改变平均导通时间，不能保证亮度线性。ADC 输入有量程、饱和、噪声和校准误差。经典 ESP32 的 GPIO34 是 ADC1_CH6，只能输入、无内部上拉。选 ADC1 避免后续 Wi-Fi 与经典 ESP32 ADC2 的资源限制。

先预测、运行和记录，再补概念。每节开始保存现有工程差异，后续沿用同一工程。

## 可选：Notebook 示例与证据卡

使用课程 .venv kernel 运行下格；这只是显示工具，不需要学习 Python。侧边栏学习不依赖这些格，build/flash/monitor 都在 ESP-IDF Terminal 执行。

In [ ]:
from pathlib import Path
import sys
import json

course_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "step2/course.json").is_file():
        course_root = candidate
        break
    link = candidate / ".academy/course.json"
    if link.is_file():
        course_root = Path(json.loads(link.read_text(encoding="utf-8"))["course_root"])
        break
if course_root is None or not (course_root / "step2/course.json").is_file():
    raise RuntimeError("请从课程工作区运行，或先在学习工程 .academy/course.json 链接课程根目录")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from tools.esp32_notebook import install, hardware_checklist
install(get_ipython())


## 把电位器限制在量程内 · Guided

断电连接 10 kΩ 电位器：两端接 3.3 V/GND，滑动端接 GPIO34；LED 仍在 GPIO18 经 330 Ω 到 GND。检查滑动端和端点不要接错，不将 5 V 接 ADC。
读 ../BOARD.md；0–3.3 V 不表示 ADC 在全段都精确线性。没有万用表时标记校准项未测。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 真实 PWM 与 ADC · Guided

main REQUIRES 增加 esp_driver_ledc esp_adc。示例选择 5 kHz、10-bit PWM；ADC 12-bit raw 缩放为 0–1023 duty，使用 ADC_ATTEN_DB_12。
先记录旋钮两端、中间的 raw 和 duty，再观察实物亮度。示例没有输出 mV，避免将理想比例当校准。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

示例来源：`step2/lessons/H04/examples/pwm_adc.cpp`。目标：`main/main.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES esp_driver_ledc esp_adc freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target main/main.cpp
#include "driver/ledc.h"
#include "esp_adc/adc_oneshot.h"
#include "esp_log.h"
#include "freertos/FreeRTOS.h"
#include "freertos/task.h"

extern "C" void app_main() {
    ledc_timer_config_t timer{};
    timer.speed_mode = LEDC_LOW_SPEED_MODE;
    timer.duty_resolution = LEDC_TIMER_10_BIT;
    timer.timer_num = LEDC_TIMER_0;
    timer.freq_hz = 5000;
    timer.clk_cfg = LEDC_AUTO_CLK;
    ESP_ERROR_CHECK(ledc_timer_config(&timer));
    ledc_channel_config_t channel{};
    channel.gpio_num = 18;
    channel.speed_mode = LEDC_LOW_SPEED_MODE;
    channel.channel = LEDC_CHANNEL_0;
    channel.timer_sel = LEDC_TIMER_0;
    ESP_ERROR_CHECK(ledc_channel_config(&channel));
    adc_oneshot_unit_init_cfg_t unit_cfg{};
    unit_cfg.unit_id = ADC_UNIT_1;
    adc_oneshot_unit_handle_t adc = nullptr;
    ESP_ERROR_CHECK(adc_oneshot_new_unit(&unit_cfg, &adc));
    adc_oneshot_chan_cfg_t adc_cfg{};
    adc_cfg.atten = ADC_ATTEN_DB_12;
    adc_cfg.bitwidth = ADC_BITWIDTH_12;
    // Classic ESP32 GPIO34 = ADC1 channel 6. Not portable to other chips.
    ESP_ERROR_CHECK(adc_oneshot_config_channel(adc, ADC_CHANNEL_6, &adc_cfg));
    while (true) {
        int raw = 0;
        ESP_ERROR_CHECK(adc_oneshot_read(adc, ADC_CHANNEL_6, &raw));
        const uint32_t duty = static_cast<uint32_t>(raw) * 1023 / 4095;
        ESP_ERROR_CHECK(ledc_set_duty(LEDC_LOW_SPEED_MODE, LEDC_CHANNEL_0, duty));
        ESP_ERROR_CHECK(ledc_update_duty(LEDC_LOW_SPEED_MODE, LEDC_CHANNEL_0));
        ESP_LOGI("measure", "raw=%d duty=%lu/1023 (not calibrated mV)", raw,
                 (unsigned long)duty);
        vTaskDelay(pdMS_TO_TICKS(200));
    }
}


## 分开采样与显示 · Modify

把日志周期从 200 ms 改成 1 s，保持采样周期不变；加入一个小窗口平均，比较静止旋钮时抖动。
预测滤波会降低哪一种噪声，又会增加多少响应延迟？不要为每次采样动态分配容器。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 把 raw 变成有依据的 mV · Build

按官方 ADC calibration 文档检查本芯片可用 scheme；创建校准句柄，用 adc_cali_raw_to_voltage 转换，处理校准不可用。
用万用表比较至少三个点，记录真实电压、估计 mV、误差、饱和区域；校准不可用时报告 raw，不冒充测量值。GPIO34 接近 3.3 V 时允许出现饱和。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 忘记提交 duty · Debug

暂时注释 ledc_update_duty，只保留 set_duty，预测日志和实物的不同。恢复后再用错误的分母 4096 与 4095 比较端点，解释量化范围。
给输入注入软件超范围数，检查缩放会不会溢出；这类测试在 host 做，不用过压制造 ADC 错误。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 自动亮度设置 · Build

需求：固定容量滤波，最小/最大亮度可配置，断线或无效采样有明确状态，采样失败不能被当成 0 lux；连续调整时按钮取消仍可响应。
不要声称电位器浮空可由任意 raw 唯一识别。给异常检测写出假设、边界与 host 测试，再记录实物行为。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 让灯对人有用 · Open

设计两档可切换曲线（线性/感知近似），用可比较的观察方法选一档；说明没有测光仪时能得出哪些结论。
H05 把本地输入换成外部传感器，保留采样失败和有效值的区分。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 验收与复盘

- [ ] PWM 亮度变化有实物记录
- [ ] ADC raw 与实测电压/校准可用性分别记录
- [ ] 噪声/响应延迟与输入错误有测试

分别记录 host、firmware build、flash、日志、实物与网络往返；未知保持未确认。手动卡片只是学习者记录，不自动测量，也不自动写入进度文件。

复盘：我选择了什么机制？哪个证据排除了错误假设？哪些内容还需要测量？把当前 Step、Hint level、代码版本、证据和下次起点写入 `.academy/progress.md`。

In [ ]:
hardware_checklist(
    'H04',
    [
        'PWM 亮度变化有实物记录',
        'ADC raw 与实测电压/校准可用性分别记录',
        '噪声/响应延迟与输入错误有测试',
    ],
)


## 遇到困难时

先完成一次尝试，再打开 `step2/lessons/H04/hints.md`，每次只读当前步骤的一层。完整参考与解题分析单独保存，默认路径不主动读取。

## 官方版本参考

- [api-reference/peripherals/ledc.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-reference/peripherals/ledc.html)
- [api-reference/peripherals/adc_oneshot.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-reference/peripherals/adc_oneshot.html)
- [api-reference/peripherals/adc_calibration.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-reference/peripherals/adc_calibration.html)